# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Input offline** nằm riêng theo notebook: `input/agoda-thy-4.csv` (cell ① `OFFLINE_FILE` / cell tải Sheet).

**Output** nằm trong `results/agoda/<RUN_NAME>/` — notebook này dùng `thy-4`, không ghi đè notebook khác:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

Cache warm (`results/agoda/captures/`) vẫn dùng chung giữa các notebook nên không tốn thêm thời gian warm.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda/<RUN_NAME>/ ──
RUN_NAME = "thy-4"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1aKBhtLv1IZpyA0EJmgzCVD4gICC-xXIKBRuyqH0ihU4/edit?gid=1083140588#gid=1083140588"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/agoda-thy-4.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 23 khách sạn. 5 dòng đầu:
   • Fusion Resort-3 bedroom villa — Three Bedroom Presidential Plunge Pool Loft - Spa Inclusive
   • Fusion Resort-4 bedroom villa — Four Bedroom Premium Villa with Private Pool - Spa Inclusive
   • Fusion Resort-5 bedroom villa — Five Bedroom Premium Beachfront Villa with Private Pool - Spa Inclusive
   • Hyatt Regency Danang Resort And Spa — 2 Twin Beds
   • Shilla Monogram Danang — Superior Partial Ocean View Twin Room


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda-thy-4.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)   # mỗi notebook 1 thư mục kết quả riêng
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
    # cache warm dùng CHUNG cho mọi notebook (đỡ warm lại) — chỉ kết quả là tách riêng
    capture_dir=os.path.join(ROOT, "results", "agoda", "captures"),
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 23 rows from TEMP_agoda.csv
🚀 AGODA crawl | 23 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-09-08
🦊 Camoufox ready (humanize=True geoip=True) — browser navs use anti-detect Firefox
✔️  1/23 Fusion Resort-3 bedroom villa — complete, skip
✔️  2/23 Fusion Resort-4 bedroom villa — complete, skip
✔️  3/23 Fusion Resort-5 bedroom villa — complete, skip
✔️  4/23 Hyatt Regency Danang Resort And Spa — complete, skip
✔️  5/23 Shilla Monogram Danang — complete, skip
✔️  6/23 Sheraton Grand Danang Beach Resort & Spa — complete, skip
✔️  7/23 TMS Hotel Da Nang Beach — complete, skip
✔️  8/23 Rosamia Da Nang Hotel — complete, skip
✔️  9/23 Golden Lotus Grand Da Nang — complete, skip
✔️  10/23 SALA DANANG BEACH HOTEL — complete, skip
✔️  11/23 Nesta Hotel Da Nang — complete, skip
✔️  12/23 Sheraton — complete, skip
✔️  13/23 InterContinental — complete, skip
✔️  14/23 Mường Thanh — complete, skip
✔️  15/23 Melia Vinpearl — complete, skip
✔️  16/23 Vinpearl Resort — complete, sk

'FINAL_20260903.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/thy-4/FINAL_20260903.csv — 23 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Fusion Resort-3 bedroom villa,Three Bedroom Presidential Plunge Pool Loft - ...,"31,638,564","32,852,281","31,638,564","28,823,930","22,947,414","28,823,930"
1,Fusion Resort-4 bedroom villa,Four Bedroom Premium Villa with Private Pool -...,"45,466,008","42,956,666","35,510,546","34,211,213","25,202,521","42,956,666"
2,Fusion Resort-5 bedroom villa,Five Bedroom Premium Beachfront Villa with Pri...,"66,201,058","60,197,090","66,201,058","60,197,090","60,197,090","60,197,090"
3,Hyatt Regency Danang Resort And Spa,2 Twin Beds,"5,000,000","3,700,000","4,050,000","3,850,000","3,600,000","3,600,000"
4,Shilla Monogram Danang,Superior Partial Ocean View Twin Room,"3,397,436","3,397,436","3,397,436","3,800,000","3,700,000","3,600,000"
5,Sheraton Grand Danang Beach Resort & Spa,"Guest room, 2 Twin, Bay view","4,900,000","3,600,000","3,893,742","3,500,000","4,400,000","4,700,000"
6,TMS Hotel Da Nang Beach,Premier City View Twin Room with Balcony and S...,"3,973,968","3,492,063","3,973,968","3,759,788","3,625,926","3,516,314"
7,Rosamia Da Nang Hotel,Grand Deluxe Double,"1,639,505","1,639,505","1,639,505","2,134,259","2,123,333","2,123,333"
8,Golden Lotus Grand Da Nang,Superior City View Room - Afternoon Tea Included,"1,464,709","1,514,974","1,540,106","1,716,032","1,942,222","1,816,561"
9,SALA DANANG BEACH HOTEL,Superior Double City View,"1,784,691","2,182,716","2,182,716","1,784,691","1,784,691","1,784,691"
